<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs NumPy, which the code below imports, while OpenBB is installed separately.

In [ ]:
!pip install numpy

OpenBB is left out on purpose. Installing openbb_terminal with pip pulls in pinned versions of pandas and other packages that can downgrade what you already have, so run the install command from the OpenBB docs in a fresh virtual environment first, then come back to this notebook.

## Imports and setup

We use yFinance to download futures price history, and NumPy for the running maximum inside the drawdown math.

In [ ]:
import yfinance as yf
import numpy as np

## Download futures prices from OpenBB

Pull three years of daily prices for the E-mini S&P 500 (ES), E-mini Dow (YM), and E-mini Nasdaq 100 (NQ) through OpenBB as continuous front-month series.

In [ ]:
data = yf.download(
    ["ES=F", "YM=F", "NQ=F"],
    start="2020-01-01",
    end="2022-12-31",
)

I picked 2020 through 2022 because the sample has to contain the February and March 2020 crash and the 2022 selloff. Without a real peak-to-trough decline the denominator of this ratio sits near zero, and the number either blows up or is undefined.

In [ ]:
data

## Convert prices into portfolio returns

Turn adjusted closing prices into daily percentage changes, then blend those changes into two portfolios using fixed weights.

In [ ]:
futs = data.Close.pct_change().dropna()

In [ ]:
port_1 = futs["ES=F"] * 0.60 + futs["YM=F"] * 0.10 + futs["NQ=F"] * 0.10

In [ ]:
port_2 = futs["ES=F"] * 0.90 + futs["YM=F"] * 0.15 + futs["NQ=F"] * 0.15

Every weight in port_2 is 1.5 times the matching weight in port_1, so port_1 holds 80 percent of the account in futures while port_2 runs 120 percent and borrows the difference. Both the yearly return and the worst loss grow with that leverage, and because returns compound, neither grows by exactly 1.5.

## Write the annualized return function

Compound the daily returns into one growth figure, then convert that to a per-year rate to normalize the length of the sample rather than the market conditions inside it.

In [ ]:
def ann_return(returns):
    ending_value = (returns + 1).prod()
    num_years = len(returns) / 252
    ann_return = ending_value ** (1 / num_years) - 1
    return ann_return

We divide by 252 because that's roughly the number of days US markets trade in a year. Adding daily returns instead of multiplying them ignores compounding, so the result can come out too high or too low, which is why prod() is here.

## Measure the worst loss and ratio

Track the running high of the portfolio, measure how far below that high it ever fell, and divide the yearly return by that loss.

In [ ]:
def calmar_ratio(returns):

    cumulative_returns = (returns + 1).cumprod() * 100

    max_return = np.fmax.accumulate(cumulative_returns)

    max_dd = ((cumulative_returns - max_return) / max_return).min()

    ann_ret = ann_return(returns)

    return ann_ret / abs(max_dd)

The np.fmax.accumulate call gives the highest value the account reached up to each day, so subtracting it measures the loss from a peak rather than from the start. The .min() call keeps the most negative of those daily losses, which means one stretch of the sample sets the whole denominator. A Calmar ratio of 1.0 means the annualized return equals the size of that maximum loss.

## Compare the two portfolios directly

Run both portfolios through the same two functions so we can put the yearly return next to the risk-adjusted number.

In [ ]:
ret = ann_return(port_1)
p1 = calmar_ratio(port_1)

In [ ]:
ret = ann_return(port_2)
p2 = calmar_ratio(port_2)

Note that ret is assigned twice, so after this runs it holds the port_2 figure only. Give each one its own name if you want both returns side by side. Print p1 and p2 to compare them, because a bigger return with a proportionally bigger loss leaves the ratio flat, which is the check I ran as a risk quant before agreeing a desk's gains came from skill.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.